In [ ]:
# 차트 작성 시 한글 깨짐 방지를 위한 koreanize-matplotlib 설치
!pip install koreanize-matplotlib

In [ ]:
# 라이브러리 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
from sklearn.datasets import load_diabetes
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 데이터 불러오기 및 데이터 이해
train_df = pd.read_csv('/content/sample_data/train.csv')
test_df = pd.read_csv('/content/sample_data/test.csv')

print("Train data loaded successfully. Shape:", train_df.shape)
print("Test data loaded successfully. Shape:", test_df.shape)

Train data loaded successfully. Shape: (7500, 11)
Test data loaded successfully. Shape: (7500, 10)


In [ ]:
train_df.info()
train_df.head()
train_df.describe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        7500 non-null   object 
 1   Exercise_Duration         7500 non-null   float64
 2   Body_Temperature(F)       7500 non-null   float64
 3   BPM                       7500 non-null   float64
 4   Height(Feet)              7500 non-null   float64
 5   Height(Remainder_Inches)  7500 non-null   float64
 6   Weight(lb)                7500 non-null   float64
 7   Weight_Status             7500 non-null   object 
 8   Gender                    7500 non-null   object 
 9   Age                       7500 non-null   int64  
 10  Calories_Burned           7500 non-null   float64
dtypes: float64(7), int64(1), object(3)
memory usage: 644.7+ KB


<bound method NDFrame.describe of               ID  Exercise_Duration  Body_Temperature(F)    BPM  Height(Feet)  \
0     TRAIN_0000               26.0                105.6  107.0           5.0   
1     TRAIN_0001                7.0                103.3   88.0           6.0   
2     TRAIN_0002                7.0                103.3   86.0           6.0   
3     TRAIN_0003               17.0                104.0   99.0           5.0   
4     TRAIN_0004                9.0                102.7   88.0           5.0   
...          ...                ...                  ...    ...           ...   
7495  TRAIN_7495               22.0                105.1  104.0           4.0   
7496  TRAIN_7496               20.0                105.3  104.0           5.0   
7497  TRAIN_7497                8.0                103.1   90.0           6.0   
7498  TRAIN_7498               12.0                104.4   97.0           5.0   
7499  TRAIN_7499               16.0                104.9   91.0           5.0   

      Height(Remainder_Inches)  Weight(lb)  Weight_Status Gender  Age  \
0                          9.0       154.3  Normal Weight      F   45   
1                          6.0       224.9     Overweight      M   50   
2                          3.0       218.3     Overweight      M   29   
3                          6.0       147.7  Normal Weight      F   33   
4                         10.0       169.8  Normal Weight      M   38   
...                        ...         ...            ...    ...  ...   
7495                      10.0       112.4  Normal Weight      F   75   
7496                       8.0       147.7  Normal Weight      F   21   
7497                       2.0       202.8     Overweight      M   57   
7498                       9.0       167.6     Overweight      M   35   
7499                      12.0       189.6     Overweight      M   26   

      Calories_Burned  
0               166.0  
1                33.0  
2                23.0  
3                91.0  
4                32.0  
...               ...  
7495            151.0  
7496            114.0  
7497             41.0  
7498             57.0  
7499             59.0  

[7500 rows x 11 columns]>

In [ ]:
def add_features(df) -> pd.DataFrame:
    d = df.copy()
    # ---------- 0) 타입 ----------
    num_cols = [
        "Exercise_Duration",
        "Body_Temperature(F)",
        "BPM",
        "Height(Feet)",
        "Height(Remainder_Inches)",
        "Weight(lb)",
        "Age",
    ]
    for c in num_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # ---------- 1) 단위 변환 (키, 몸무게, 체온) ----------

    # 키
    d["height_in"] = d["Height(Feet)"] * 12 + d["Height(Remainder_Inches)"]
    d["height_cm"] = d["height_in"] * 2.54
    d["height_m"] = d["height_cm"] / 100.0

    # 몸무게
    d["weight_kg"] = d["Weight(lb)"] * 0.45359237

    # 체온
    d["temp_c"] = (d["Body_Temperature(F)"] - 32) * (5 / 9)

    # ---------- 2) 성별/대사 ----------
    # BMI
    d["bmi"] = d["weight_kg"] / (d["height_m"] ** 2)

    # Gender 인코딩 (M=1, F=0; 그 외는 NaN)
    d["Gender"] = (
        d["Gender"].astype(str).str.strip().str.upper()
        .map({"M": 1, "MALE": 1, "F": 0, "FEMALE": 0})
    )
# 파생변수 생성

# ====== 신체 정보 관련 ======
# 1. BMR - 기초대사량: 해리스 - 베네딕트 계산식
    d["bmr"] = np.where(
        d["Gender"] == 1,
        66.47 + (13.75 * d["weight_kg"]) + (5.0 * d["height_cm"]) - (6.76 * d["Age"]),  # 남자
        np.where(
            d["Gender"] == 0,
            655.1 + (9.56 * d["weight_kg"]) + (1.85 * d["height_cm"]) - (4.68 * d["Age"]),  # 여자
            np.nan
        )
    )

# ====== 운동 관련 ======
# 2. MHR: 최대 심박수 계산
    d["mhr"] = 220 - d["Age"]

# 3. activity_intensity: 활동 강도 계산
    d["activity_intensity"] = get_activity_intensity_by_age(d["BPM"], d["Age"])

# 4. 심박 예비력 비율 계산 Heart Rate Reserve
    HR_REST = 60  # 안정 시 심박수 계산에 사용되는 기본 심박수
    d["hrr"] = d["mhr"] - HR_REST  # 내가 낼 수 있는 최대 심박 중에서 지금 얼마나 쓰고 있는지 나타내는 지표
    d["hrr_ratio"] = (d["BPM"] - HR_REST) / d["hrr"]
# 강도가 높을수록 비율이 높아짐 -> 퍼센티지!! 1에 가까울수록 높은 강도

# 5. 열부하지표 (운동 시 체온 상승은 대사를 증가시킴)
# 체온이 1도씨 상승하면 대사율 약 10-13% 증가
    d["thermal_load"] = (d["temp_c"] - 37) * d["Exercise_Duration"]

# ====== 신체와 운동 합친 것 관련 ======

# 6. 체중 보정 심박 부하
    d["relative_workload"] = (d["BPM"] * d["weight_kg"] * d["Exercise_Duration"]) / d["Age"]

# 7. 심박 스트레스 비율 지수
    d["hb_stress"] = d["BPM"] / d["mhr"]

# 8. 대사 스트레스 지수
    d["metabolic_stress"] = d["BPM"] * d["Exercise_Duration"]
    return d


def get_activity_intensity_by_age(bpm_series, age_series):
    """
    연령별 목표 심박수 표 기반으로 Light/Moderate/High 분류

Light: bpm < low_cut
Moderate: low_cut <= bpm < high_cut
High: bpm >= high_cut
"""

    # 표(이미지) 기반 앵커 포인트
    ages = np.array([15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,95], dtype=float)

    # 저강도 경계(표의 '<' 값)
    low_cut_points = np.array([126,124,122,120,117,115,113,111,109,107,105,102,100,98,96,92], dtype=float)

    # 중강도 상한(표의 '중강도' 범위의 상단 값)
    high_cut_points = np.array([150,147,145,142,139,137,134,132,129,127,124,122,119,117,114,109], dtype=float)

    age = pd.to_numeric(age_series, errors="coerce").astype(float).to_numpy()
    bpm = pd.to_numeric(bpm_series, errors="coerce").astype(float).to_numpy()

    # 나이에 맞춰 컷을 선형 보간 (15 미만/95 초과는 끝값으로 고정)
    low_cut  = np.interp(age, ages, low_cut_points)
    high_cut = np.interp(age, ages, high_cut_points)

    out = np.where(bpm < low_cut, "Light",
          np.where(bpm < high_cut, "Moderate", "High"))


    return out


df_fe = add_features(train_df)
display(df_fe)

,ID,Exercise_Duration,Body_Temperature(F),BPM,Height(Feet),Height(Remainder_Inches),Weight(lb),Weight_Status,Gender,Age,...,bmi,bmr,mhr,activity_intensity,hrr,hrr_ratio,thermal_load,relative_workload,hb_stress,metabolic_stress
0,TRAIN_0000,26.0,105.6,107.0,5.0,9.0,154.3,Normal Weight,0,45,...,22.785893,1437.828734,175,Light,115,0.408696,101.111111,4326.894224,0.611429,2782.0
1,TRAIN_0001,7.0,103.3,88.0,6.0,6.0,224.9,Overweight,1,50,...,25.989538,2121.747705,170,Light,110,0.254545,18.277778,1256.799224,0.517647,616.0
2,TRAIN_0002,7.0,103.3,86.0,6.0,3.0,218.3,Overweight,1,29,...,27.285349,2184.444198,191,Light,131,0.198473,18.277778,2055.502312,0.450262,602.0
3,TRAIN_0003,17.0,104.0,99.0,5.0,6.0,147.7,Normal Weight,0,33,...,23.839159,1451.271870,187,Light,127,0.307087,51.000000,3416.775245,0.529412,1683.0
4,TRAIN_0004,9.0,102.7,88.0,5.0,10.0,169.8,Normal Weight,1,38,...,24.363513,1757.614786,182,Light,122,0.229508,20.500000,1605.258623,0.483516,792.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7495,TRAIN_7495,22.0,105.1,104.0,4.0,10.0,112.4,Normal Weight,0,75,...,23.491385,1064.046960,145,Moderate,85,0.517647,79.444444,1555.345255,0.717241,2288.0
7496,TRAIN_7496,20.0,105.3,104.0,5.0,8.0,147.7,Normal Weight,0,21,...,22.457478,1516.829870,199,Light,139,0.316547,74.444444,6635.753978,0.522613,2080.0
7497,TRAIN_7497,8.0,103.1,90.0,6.0,2.0,202.8,Overweight,1,57,...,26.037712,1885.792324,163,Light,103,0.291262,20.000000,1161.960412,0.552147,720.0
7498,TRAIN_7498,12.0,104.4,97.0,5.0,9.0,167.6,Overweight,1,35,...,24.749939,1751.473617,185,Light,125,0.296000,38.666667,2528.277215,0.524324,1164.0


In [ ]:
bins = [0, 18.5, 23, 25, 30, 100]
labels = ["저체중", "정상", "과체중", "비만", "고도비만"]
df_fe['BMI_Category'] = pd.cut(df_fe['bmi'], bins=bins, labels=labels, right=False)

mapping = {
    "저체중": 0,
    "정상": 1,
    "과체중": 2,
    "비만": 3,
    "고도비만":4
}

df_fe["BMI_Category"] = df_fe["BMI_Category"].map(mapping)

display(df_fe.head())


,ID,Exercise_Duration,Body_Temperature(F),BPM,Height(Feet),Height(Remainder_Inches),Weight(lb),Weight_Status,Gender,Age,...,bmr,mhr,activity_intensity,hrr,hrr_ratio,thermal_load,relative_workload,hb_stress,metabolic_stress,BMI_Category
0,TRAIN_0000,26.0,105.6,107.0,5.0,9.0,154.3,Normal Weight,0,45,...,1437.828734,175,Light,115,0.408696,101.111111,4326.894224,0.611429,2782.0,1
1,TRAIN_0001,7.0,103.3,88.0,6.0,6.0,224.9,Overweight,1,50,...,2121.747705,170,Light,110,0.254545,18.277778,1256.799224,0.517647,616.0,3
2,TRAIN_0002,7.0,103.3,86.0,6.0,3.0,218.3,Overweight,1,29,...,2184.444198,191,Light,131,0.198473,18.277778,2055.502312,0.450262,602.0,3
3,TRAIN_0003,17.0,104.0,99.0,5.0,6.0,147.7,Normal Weight,0,33,...,1451.271870,187,Light,127,0.307087,51.000000,3416.775245,0.529412,1683.0,2
4,TRAIN_0004,9.0,102.7,88.0,5.0,10.0,169.8,Normal Weight,1,38,...,1757.614786,182,Light,122,0.229508,20.500000,1605.258623,0.483516,792.0,2


In [ ]:
bins_age = [20, 30, 40, 50, 60, 70, 80]
labels_age = ["20대", "30대", "40대", "50대", "60대", "70대"]
df_fe["연령대"] = pd.cut(df_fe["Age"], bins=bins_age, labels=labels_age, right=False)
mapping = {
    "20대": 0,
    "30대": 1,
    "40대": 2,
    "50대": 3,
    "60대":4,
    "70대":5,
}

df_fe["연령대"] = df_fe["연령대"].map(mapping)

display(df_fe.head())

,ID,Exercise_Duration,Body_Temperature(F),BPM,Height(Feet),Height(Remainder_Inches),Weight(lb),Weight_Status,Gender,Age,...,mhr,activity_intensity,hrr,hrr_ratio,thermal_load,relative_workload,hb_stress,metabolic_stress,BMI_Category,연령대
0,TRAIN_0000,26.0,105.6,107.0,5.0,9.0,154.3,Normal Weight,0,45,...,175,Light,115,0.408696,101.111111,4326.894224,0.611429,2782.0,1,2
1,TRAIN_0001,7.0,103.3,88.0,6.0,6.0,224.9,Overweight,1,50,...,170,Light,110,0.254545,18.277778,1256.799224,0.517647,616.0,3,3
2,TRAIN_0002,7.0,103.3,86.0,6.0,3.0,218.3,Overweight,1,29,...,191,Light,131,0.198473,18.277778,2055.502312,0.450262,602.0,3,0
3,TRAIN_0003,17.0,104.0,99.0,5.0,6.0,147.7,Normal Weight,0,33,...,187,Light,127,0.307087,51.000000,3416.775245,0.529412,1683.0,2,1
4,TRAIN_0004,9.0,102.7,88.0,5.0,10.0,169.8,Normal Weight,1,38,...,182,Light,122,0.229508,20.500000,1605.258623,0.483516,792.0,2,1


In [ ]:
# 변수 가공

# 기존 변수에서 'ID' 'Height(Remainder_Inches)', 'Weight(lb)', 'Body_Temperature(F)' 삭제
df_fe = df_fe.drop(columns=['height_in','height_cm','height_m','ID', 'Height(Feet)','Height(Remainder_Inches)', 'Weight(lb)', 'Body_Temperature(F)', 'Weight_Status']) # 학습데이터
# 최종 DataFrame 확인
display(df_fe.head())

,Exercise_Duration,BPM,Gender,Age,Calories_Burned,weight_kg,temp_c,bmi,bmr,mhr,activity_intensity,hrr,hrr_ratio,thermal_load,relative_workload,hb_stress,metabolic_stress,BMI_Category,연령대
0,26.0,107.0,0,45,166.0,69.989303,40.888889,22.785893,1437.828734,175,Light,115,0.408696,101.111111,4326.894224,0.611429,2782.0,1,2
1,7.0,88.0,1,50,33.0,102.012924,39.611111,25.989538,2121.747705,170,Light,110,0.254545,18.277778,1256.799224,0.517647,616.0,3,3
2,7.0,86.0,1,29,23.0,99.019214,39.611111,27.285349,2184.444198,191,Light,131,0.198473,18.277778,2055.502312,0.450262,602.0,3,0
3,17.0,99.0,0,33,91.0,66.995593,40.000000,23.839159,1451.271870,187,Light,127,0.307087,51.000000,3416.775245,0.529412,1683.0,2,1
4,9.0,88.0,1,38,32.0,77.019984,39.277778,24.363513,1757.614786,182,Light,122,0.229508,20.500000,1605.258623,0.483516,792.0,2,1


In [ ]:
# 결측치(NaN) 확인
missing_values = df_fe.isnull().sum()
display(missing_values)

,0
Exercise_Duration,0
BPM,0
Gender,0
Age,0
Calories_Burned,0
weight_kg,0
temp_c,0
bmi,0
bmr,0
mhr,0


In [ ]:
# 숨겨진 결측치 (0값) 탐색
zero_cols = ['Exercise_Duration', 'BPM','Age','temp_c','bmi']

for col in zero_cols:
    zero_count = (df_fe[col] == 0).sum()
    print(f" - {col}: {zero_count}개 ({zero_count/len(df_fe)*100:.2f}%)")

 - Exercise_Duration: 0개 (0.00%)
 - BPM: 0개 (0.00%)
 - Age: 0개 (0.00%)
 - temp_c: 0개 (0.00%)
 - bmi: 0개 (0.00%)


In [ ]:
# 수치형 데이터 요약
df_fe.describe()

,Exercise_Duration,BPM,Gender,Age,Calories_Burned,weight_kg,temp_c,bmi,bmr,mhr,hrr,hrr_ratio,thermal_load,relative_workload,hb_stress,metabolic_stress
count,7500.0000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000
mean,15.5012,95.498133,0.497467,42.636000,89.373467,75.006573,40.018652,24.344072,1628.768235,177.364000,117.364000,0.309697,52.722600,3159.374777,0.543725,1548.695333
std,8.3553,9.587331,0.500027,16.883188,62.817086,15.108316,0.784914,1.573061,315.148409,16.883188,16.883188,0.098758,34.794704,2469.806426,0.078308,931.651745
min,1.0000,69.000000,0.000000,20.000000,1.000000,36.015234,37.111111,19.520854,968.277458,141.000000,81.000000,0.066667,0.111111,72.935560,0.353846,70.000000
25%,8.0000,88.000000,0.000000,28.000000,35.000000,63.003980,39.611111,23.146157,1366.719460,164.000000,104.000000,0.236842,21.777778,1250.318053,0.485549,728.000000
50%,15.0000,95.000000,0.000000,39.000000,77.000000,73.980916,40.222222,24.363513,1552.320438,181.000000,121.000000,0.301724,49.166667,2599.006838,0.535948,1456.000000
75%,23.0000,103.000000,1.000000,56.000000,138.000000,86.999017,40.611111,25.481072,1892.439755,192.000000,132.000000,0.370968,83.055556,4391.456667,0.591549,2323.000000
max,30.0000,128.000000,1.000000,79.000000,300.000000,131.995380,41.500000,29.414135,2703.206470,200.000000,140.000000,0.684211,130.500000,16537.388140,0.809859,3840.000000


In [ ]:
df_fe['activity_intensity'] = df_fe['activity_intensity'].astype('category')
df_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Exercise_Duration   7500 non-null   float64 
 1   BPM                 7500 non-null   float64 
 2   Gender              7500 non-null   int64   
 3   Age                 7500 non-null   int64   
 4   Calories_Burned     7500 non-null   float64 
 5   weight_kg           7500 non-null   float64 
 6   temp_c              7500 non-null   float64 
 7   bmi                 7500 non-null   float64 
 8   bmr                 7500 non-null   float64 
 9   mhr                 7500 non-null   int64   
 10  activity_intensity  7500 non-null   category
 11  hrr                 7500 non-null   int64   
 12  hrr_ratio           7500 non-null   float64 
 13  thermal_load        7500 non-null   float64 
 14  relative_workload   7500 non-null   float64 
 15  hb_stress           7500 non-null   fl

In [ ]:
mapping = {
    "Light": 0,
    "Moderate": 1,
    "High": 2
    }

df_fe["activity_intensity"] = df_fe["activity_intensity"].map(mapping)

In [ ]:
#  최종 문제지
df_fe.head()

,Exercise_Duration,BPM,Gender,Age,Calories_Burned,weight_kg,temp_c,bmi,bmr,mhr,activity_intensity,hrr,hrr_ratio,thermal_load,relative_workload,hb_stress,metabolic_stress,BMI_Category,연령대
0,26.0,107.0,0,45,166.0,69.989303,40.888889,22.785893,1437.828734,175,0,115,0.408696,101.111111,4326.894224,0.611429,2782.0,1,2
1,7.0,88.0,1,50,33.0,102.012924,39.611111,25.989538,2121.747705,170,0,110,0.254545,18.277778,1256.799224,0.517647,616.0,3,3
2,7.0,86.0,1,29,23.0,99.019214,39.611111,27.285349,2184.444198,191,0,131,0.198473,18.277778,2055.502312,0.450262,602.0,3,0
3,17.0,99.0,0,33,91.0,66.995593,40.000000,23.839159,1451.271870,187,0,127,0.307087,51.000000,3416.775245,0.529412,1683.0,2,1
4,9.0,88.0,1,38,32.0,77.019984,39.277778,24.363513,1757.614786,182,0,122,0.229508,20.500000,1605.258623,0.483516,792.0,2,1


In [ ]:
# 수치형 데이터 요약
df_fe.describe()

,Exercise_Duration,BPM,Gender,Age,Calories_Burned,weight_kg,temp_c,bmi,bmr,mhr,hrr,hrr_ratio,thermal_load,relative_workload,hb_stress,metabolic_stress
count,7500.0000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000
mean,15.5012,95.498133,0.497467,42.636000,89.373467,75.006573,40.018652,24.344072,1628.768235,177.364000,117.364000,0.309697,52.722600,3159.374777,0.543725,1548.695333
std,8.3553,9.587331,0.500027,16.883188,62.817086,15.108316,0.784914,1.573061,315.148409,16.883188,16.883188,0.098758,34.794704,2469.806426,0.078308,931.651745
min,1.0000,69.000000,0.000000,20.000000,1.000000,36.015234,37.111111,19.520854,968.277458,141.000000,81.000000,0.066667,0.111111,72.935560,0.353846,70.000000
25%,8.0000,88.000000,0.000000,28.000000,35.000000,63.003980,39.611111,23.146157,1366.719460,164.000000,104.000000,0.236842,21.777778,1250.318053,0.485549,728.000000
50%,15.0000,95.000000,0.000000,39.000000,77.000000,73.980916,40.222222,24.363513,1552.320438,181.000000,121.000000,0.301724,49.166667,2599.006838,0.535948,1456.000000
75%,23.0000,103.000000,1.000000,56.000000,138.000000,86.999017,40.611111,25.481072,1892.439755,192.000000,132.000000,0.370968,83.055556,4391.456667,0.591549,2323.000000
max,30.0000,128.000000,1.000000,79.000000,300.000000,131.995380,41.500000,29.414135,2703.206470,200.000000,140.000000,0.684211,130.500000,16537.388140,0.809859,3840.000000
